In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from custom_stopwords import academic_stopwords_extended

In [17]:
scopus_df = pd.read_csv(r"abstracts_sample.csv", low_memory=False)

In [18]:
clean_scopus_df = scopus_df[['Abstract', 'Year']].copy()
clean_scopus_df = clean_scopus_df.dropna(subset =['Abstract'])

In [19]:
clean_scopus_df['Year'] = pd.to_numeric(clean_scopus_df['Year'], errors='coerce')
clean_scopus_df = clean_scopus_df.dropna(subset=['Year'])
clean_scopus_df['Year'] = clean_scopus_df['Year'].astype(int)
clean_scopus_df = clean_scopus_df.reset_index(drop=True)

In [21]:
clean_scopus_df.shape

(200, 2)

In [ ]:
custom_stop_words = list(ENGLISH_STOP_WORDS) + academic_stopwords_extended

In [23]:
vectorizer = TfidfVectorizer(max_features=1000,
                             stop_words=custom_stop_words,
                             max_df = 0.25,
                             min_df = 6,
                             token_pattern=r'(?u)\b[a-zA-Z]{2,}\b')

In [24]:
tfidf_matrix = vectorizer.fit_transform(clean_scopus_df['Abstract'])

In [25]:
tfidf_matrix.shape

(200, 192)

In [27]:
past_df = clean_scopus_df[clean_scopus_df['Year'] <= 2022].copy()
present_df = clean_scopus_df[clean_scopus_df['Year'] > 2022].copy()

In [28]:
past_indices = past_df.index
present_indices = present_df.index

In [29]:
past_matrix = tfidf_matrix[past_indices]
present_matrix = tfidf_matrix[present_indices]

In [30]:
past_centroid = np.asarray(past_matrix.mean(axis=0))
present_centroid = np.asarray(present_matrix.mean(axis=0))

In [31]:
similarity_matrix = cosine_similarity(past_centroid, present_centroid)
result = similarity_matrix[0][0] * 100

In [32]:
print(f"{result:.2f}%")

84.85%
